# DFA Simulator
### M = (Q, Σ, δ, q0, F)

This notebook implements a complete Deterministic Finite Automaton (DFA) Simulator.

**Modules:**
- `DFA` / `DFATransition` — Core data structure
- `DFAValidator` — Mathematical correctness checks
- `DFABuilder` — Parse raw text input into a DFA object
- `DFARunner` — Step-by-step string simulation

In [ ]:
from dataclasses import dataclass

DEAD = 'DEAD'

@dataclass
class DFATransition:
    from_state: str
    symbol: str
    to_state: str

    def __repr__(self):
        return f"delta({self.from_state}, '{self.symbol}') -> {self.to_state}"


class DFA:
    def __init__(self, states, alphabet, start_state, final_states, transitions):
        self.states = set(states)
        self.alphabet = set(alphabet)
        self.start_state = start_state
        self.final_states = set(final_states)
        self.transitions = list(transitions)

        self.delta = {state: {} for state in self.states}
        for t in self.transitions:
            self.delta[t.from_state][t.symbol] = t.to_state

        self._complete_with_dead()
        self._unreachable = None
        self._dead_states = None

    def _complete_with_dead(self):
        needs_dead = False
        for state in list(self.states):
            for symbol in self.alphabet:
                if symbol not in self.delta[state]:
                    needs_dead = True
                    self.delta[state][symbol] = DEAD
        if needs_dead:
            self.states.add(DEAD)
            self.delta[DEAD] = {symbol: DEAD for symbol in self.alphabet}
            for symbol in self.alphabet:
                self.transitions.append(DFATransition(DEAD, symbol, DEAD))

    def has_dead_state(self):
        return DEAD in self.states

    def get_next_state(self, state, symbol):
        return self.delta.get(state, {}).get(symbol, None)

    def is_accepting(self, state):
        return state in self.final_states

    def get_defined_transitions(self):
        return self.transitions

    def get_unreachable_states(self):
        if self._unreachable is not None:
            return self._unreachable
        visited = set()
        queue = [self.start_state]
        visited.add(self.start_state)
        while queue:
            current = queue.pop(0)
            for symbol in self.alphabet:
                nxt = self.get_next_state(current, symbol)
                if nxt and nxt not in visited:
                    visited.add(nxt)
                    queue.append(nxt)
        self._unreachable = self.states - visited
        return self._unreachable

    def get_dead_states(self):
        if self._dead_states is not None:
            return self._dead_states
        reverse = {s: set() for s in self.states}
        for state in self.states:
            for symbol in self.alphabet:
                nxt = self.get_next_state(state, symbol)
                if nxt:
                    reverse[nxt].add(state)
        can_reach_accept = set()
        queue = []
        for s in self.final_states:
            if s in self.states:
                queue.append(s)
                can_reach_accept.add(s)
        while queue:
            current = queue.pop(0)
            for predecessor in reverse.get(current, set()):
                if predecessor not in can_reach_accept:
                    can_reach_accept.add(predecessor)
                    queue.append(predecessor)
        self._dead_states = self.states - can_reach_accept
        return self._dead_states

    def is_language_empty(self):
        reachable = self.states - self.get_unreachable_states()
        return len(reachable & self.final_states) == 0

    def __repr__(self):
        return (f'DFA(\n  Q  = {self.states}\n  Σ  = {self.alphabet}\n'
                f'  q0 = {self.start_state}\n  F  = {self.final_states}\n)')

print('✔ Module 1 loaded: DFA, DFATransition')

✔ Module 1 loaded: DFA, DFATransition


---
## Module 2 — validator.py : DFA Validation

Checks 6 rules before building the DFA:
1. Start state must be in Q
2. All final states must be in Q
3. Every transition from_state must be in Q
4. Every transition to_state must be in Q
5. Every transition symbol must be in Σ
6. No duplicate (state, symbol) pair — enforces DFA determinism

Returns (is_valid, list_of_errors).

In [ ]:
from dataclasses import dataclass as _dc

@_dc
class ValidationError:
    rule: str
    message: str
    def __str__(self):
        return f'[{self.rule}] {self.message}'


class DFAValidator:
    def validate(self, states, alphabet, start_state, final_states, transitions):
        errors = []
        self._check_start_state(start_state, states, errors)
        self._check_final_states(final_states, states, errors)
        self._check_transitions(transitions, states, alphabet, errors)
        return len(errors) == 0, errors

    def _check_start_state(self, start_state, states, errors):
        if start_state not in states:
            errors.append(ValidationError('START STATE',
                f"Start state '{start_state}' is not in states {states}."))

    def _check_final_states(self, final_states, states, errors):
        for state in final_states:
            if state not in states:
                errors.append(ValidationError('FINAL STATES',
                    f"Final state '{state}' is not in states {states}."))

    def _check_transitions(self, transitions, states, alphabet, errors):
        seen = set()
        for from_state, symbol, to_state in transitions:
            if from_state not in states:
                errors.append(ValidationError('TRANSITION FROM',
                    f"from_state '{from_state}' not in states {states}."))
            if to_state not in states:
                errors.append(ValidationError('TRANSITION TO',
                    f"to_state '{to_state}' not in states {states}."))
            if symbol not in alphabet:
                errors.append(ValidationError('TRANSITION SYMBOL',
                    f"symbol '{symbol}' not in alphabet {alphabet}."))
            key = (from_state, symbol)
            if key in seen:
                errors.append(ValidationError('DETERMINISM',
                    f"Duplicate transition for ('{from_state}', '{symbol}')."))
            else:
                seen.add(key)

print('✔ Module 2 loaded: DFAValidator')

✔ Module 2 loaded: DFAValidator


---
## Module 3 — builder.py : Input Parser and DFA Builder

Parses raw text into a DFA in two steps:
- Step 1: _parse() reads each labeled section line by line
- Step 2: validate() checks correctness, then builds and returns the DFA

In [ ]:
class ParseError(Exception):
    pass


class DFABuilder:
    def __init__(self):
        self._validator = DFAValidator()

    def build(self, raw_input):
        states, alphabet, start_state, final_states, raw_transitions = self._parse(raw_input)
        is_valid, errors = self._validator.validate(
            states, alphabet, start_state, final_states, raw_transitions)
        if not is_valid:
            self._report_errors(errors)
            return None
        transitions = [DFATransition(f, s, t) for f, s, t in raw_transitions]
        return DFA(states, alphabet, start_state, final_states, transitions)

    def _parse(self, raw_input):
        lines = [l.strip() for l in raw_input.strip().splitlines() if l.strip()]
        states       = self._parse_field(lines, 'States:')
        alphabet     = self._parse_field(lines, 'Alphabet:')
        start_state  = self._parse_single(lines, 'Start state:')
        final_states = self._parse_field(lines, 'Final states:')
        n            = self._parse_count(lines, 'Number of transitions:')
        transitions  = self._parse_transitions(lines, n)
        return set(states), set(alphabet), start_state, set(final_states), transitions

    def _parse_field(self, lines, prefix):
        for line in lines:
            if line.lower().startswith(prefix.lower()):
                value = line[len(prefix):].strip()
                if not value:
                    raise ParseError(f"'{prefix}' section is empty.")
                return value.split()
        raise ParseError(f"Missing required section: '{prefix}'")

    def _parse_single(self, lines, prefix):
        values = self._parse_field(lines, prefix)
        if len(values) != 1:
            raise ParseError(f"'{prefix}' must have exactly one value.")
        return values[0]

    def _parse_count(self, lines, prefix):
        values = self._parse_field(lines, prefix)
        if len(values) != 1 or not values[0].isdigit():
            raise ParseError(f"'{prefix}' must be a single non-negative integer.")
        return int(values[0])

    def _parse_transitions(self, lines, n):
        skip = ('states:','alphabet:','start state:','final states:','number of transitions:')
        trans_lines = [l for l in lines if not any(l.lower().startswith(p) for p in skip)]
        if len(trans_lines) < n:
            raise ParseError(f"Expected {n} transitions, found {len(trans_lines)}.")
        result = []
        for line in trans_lines[:n]:
            parts = line.split()
            if len(parts) != 3:
                raise ParseError(f"Bad transition format: '{line}'.")
            result.append((parts[0], parts[1], parts[2]))
        return result

    def _report_errors(self, errors):
        print('\nInvalid DFA — errors found:\n')
        for i, e in enumerate(errors, 1):
            print(f'  {i}. {e}')
        print('\nPlease fix the above and try again.\n')

print('✔ Module 3 loaded: DFABuilder')

✔ Module 3 loaded: DFABuilder


---
## Module 4 — runner.py : Step-by-Step Simulation

Simulates the DFA on an input string one symbol at a time:
1. Start at q0
2. For each symbol: if not in alphabet → reject immediately
3. Look up δ(current, symbol) → move, print the step
4. If DEAD state reached → halt early, reject
5. After all symbols: if current state ∈ F → Accept, else Reject

In [ ]:
class DFARunner:
    def run(self, dfa, input_string):
        print(f"\nInput string: {input_string if input_string else '(empty)'}")
        current_state = dfa.start_state
        print(f'Start at state: {current_state}')

        for symbol in input_string:
            if symbol not in dfa.alphabet:
                print(f"Read '{symbol}' -> not in alphabet. Rejected.")
                return False
            next_state = dfa.get_next_state(current_state, symbol)
            print(f"Read '{symbol}' -> move from {current_state} to {next_state}")
            current_state = next_state
            if current_state == DEAD:
                print('Entered DEAD state. Execution halted early.')
                print('Result: Rejected')
                return False

        accepted = dfa.is_accepting(current_state)
        print(f'Halted at state: {current_state}')
        print('Result: Accepted' if accepted else 'Result: Rejected')
        return accepted

print('✔ Module 4 loaded: DFARunner')

✔ Module 4 loaded: DFARunner


---
## Module 5 — Analysis Helper

Prints unreachable states, dead states, DEAD trap state info, and empty language check.

In [ ]:
def print_analysis(dfa):
    sep = '-' * 50
    print(sep)
    print('  DFA Analysis')
    print(sep)

    unreachable = dfa.get_unreachable_states() - {DEAD}
    if unreachable:
        print(f"  ⚠  Unreachable states : {sorted(unreachable)}")
        print( '     (Never reachable from start state)')
    else:
        print('  Unreachable states : none')

    dead = dfa.get_dead_states() - {DEAD}
    if dead:
        print(f"  ⚠  Dead states        : {sorted(dead)}")
        print( '     (Cannot reach any final state from here)')
    else:
        print('  Dead states        : none')

    if dfa.has_dead_state():
        print("  DEAD trap state added: undefined transitions -> 'DEAD'")

    if dfa.is_language_empty():
        print('  ⚠  Language is EMPTY — no string is accepted.')
    else:
        print('  Language : non-empty (at least one string accepted).')

    print(sep)

print('✔ Module 5 loaded: print_analysis')

---
## Demo — Build and Run

Edit DFA_INPUT and TEST_STRINGS below, then run the cell.

In [ ]:
print("Enter your DFA definition (press Enter on a blank line when done):")
print()

lines = []
while True:
    line = input()
    if line.strip() == "":
        break
    lines.append(line)

raw_input = "\n".join(lines)

builder = DFABuilder()
try:
    dfa = builder.build(raw_input)
except ParseError as e:
    print(f'Parse error: {e}')
    dfa = None

if dfa:
    print('\n✔ DFA built successfully!\n')
    print_analysis(dfa)
    print('\n' + '-'*50)
    print('  Simulations')
    print('-'*50)
    runner = DFARunner()
    print("\nEnter strings to test one by one (press Enter on a blank line to stop):")
    print()
    while True:
        s = input("String to test: ")
        if s.strip() == "":
            break
        runner.run(dfa, s)